In [1]:
import pandas as pd
import joblib
import time

from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV  # Bayesian optimization: utilizado para optimizar hiperparámetros

import lightgbm as lgbm
from lightgbm import early_stopping  # Early stopping: utilizado para evitar sobreajuste

from Funcoes_Comuns import avaliar_modelo, registrar_modelo

### 1. Recuperar base já pré-processada

In [2]:
# Obter dados socioeconômicos e notas do ENEM 2023
df_enem = pd.read_pickle('Bases\\Finais\\dados_escolares_2023.pkl')

In [ ]:
variaveis_alvo = ['NUM_NOTA_MT', 'NUM_NOTA_LC', 'NUM_NOTA_CN', 'NUM_NOTA_CH', 'NUM_NOTA_REDACAO']
grupo_previsao = ['NUM_NOTA_CH']

# separar em treino e teste
X = df_enem.drop(columns=variaveis_alvo)
y = df_enem[grupo_previsao]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ajuste de tipo para MLflow -> Converter colunas inteiras para float
X_train = X_train.astype({col: 'float' for col in X_train.select_dtypes('int').columns})
X_test = X_test.astype({col: 'float' for col in X_test.select_dtypes('int').columns})

# Obter colunas categóricas
categorical_features = X_train.select_dtypes(include=['category']).columns.tolist()

# Criar Eval Set para validação cruzada (15% do conjunto de treino)
X_train_final, X_eval, y_train_final, y_eval = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42
)

In [4]:
# Ajustar as dimensões dos arrays
y_test = y_test.squeeze()
y_train_final = y_train_final.squeeze()
y_eval = y_eval.squeeze()

#### 2. Modelo Base

In [5]:
# Treinar modelo LGBMRegressor Base
modelo_lgbm = lgbm.LGBMRegressor(n_estimators=1000, 
                                 learning_rate=0.01, 
                                 random_state=42,
                                 max_bin=4095,
                                 force_row_wise=True)

start_time = time.time()

modelo_lgbm.fit(X_train_final, 
                y_train_final,
                eval_set=[(X_eval, y_eval)],
                eval_metric=['r2', 'rmse', 'mae'],
                callbacks=[early_stopping(stopping_rounds=200)],
                categorical_feature=categorical_features)

tempo_treino = time.time() - start_time

[LightGBM] [Info] Total Bins 30558
[LightGBM] [Info] Number of data points in the train set: 487214, number of used features: 73
[LightGBM] [Info] Start training from score 527.881246
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[613]	valid_0's rmse: 74.2043	valid_0's l1: 59.0162	valid_0's l2: 5506.28


In [6]:
# Previsões
y_pred = modelo_lgbm.predict(X_test)

In [7]:
nome_experimento = 'Escolares CH 2023'

registrar_modelo(experimento=nome_experimento,
                 parametros={**modelo_lgbm.get_params(), "amostra": X_train_final.shape[0], "tempo": tempo_treino},
                 X_train=X_train_final,
                 y_train=y_train_final,
                 y_test=y_test,
                 y_pred=y_pred,
                 variavel_alvo='NUM_NOTA_CH',
                 modelo=modelo_lgbm,
                 nome_modelo='modelo_lgbm_base_escolares_2023',
                 descricao_modelo='Modelo LGBMRegressor base para dados escolares CH',)

2025/08/11 21:13:05 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
Registered model 'modelo_lgbm_base_escolares_2023' already exists. Creating a new version of this model...
2025/08/11 21:13:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: modelo_lgbm_base_escolares_2023, version 3


Modelo registrado com sucesso no MLflow: modelo_lgbm_base_escolares_2023
🏃 View run unequaled-fox-609 at: http://127.0.0.1:9080/#/experiments/623589849857672017/runs/af488f8dd70940ecad041c4592c723d0
🧪 View experiment at: http://127.0.0.1:9080/#/experiments/623589849857672017
Rastreamento do MLflow finalizado.


Created version '3' of model 'modelo_lgbm_base_escolares_2023'.


In [8]:
# Avaliação grupo treino
avaliar_modelo(y_train_final, modelo_lgbm.predict(X_train_final), "treino")

# Avaliação grupo teste
avaliar_modelo(y_test, y_pred, "teste")

MAE (treino): 58.3898
RMSE (treino): 73.5343
R2 (treino): 0.2451
MAE (teste): 58.7456
RMSE (teste): 73.9774
R2 (teste): 0.2323


#### 3. Bayes Search

In [9]:
modelo_lgbm_bayes = lgbm.LGBMRegressor(random_state=42,
                                       max_bin=4095, 
                                       force_row_wise=True)

In [10]:
# Definição do espaço de busca para otimização bayesiana
param_grid = {
    'num_leaves': (5, 60),                         # Número de folhas na árvore de decisão
    'max_depth': (60, 120),                        # Profundidade máxima da árvore
    'learning_rate': (0.001, 0.01, 'log-uniform'), # Taxa de aprendizado
    'n_estimators': (5000, 8000),                  # Número de árvores
    'subsample': (0.1, 0.9),                       # Proporção de amostras usadas em cada árvore
    'colsample_bytree': (0.1, 0.9),                # Fração de colunas a serem usadas por árvore
    'reg_alpha': (1e-3, 1.0, 'log-uniform'),       # Regularização L1
    'reg_lambda': (1e-7, 1e-2, 'log-uniform'),     # Regularização L2
}

In [11]:
# Configurar a busca Bayesiana usando BayesSearchCV

# Criando o otimizador Bayesiano
bayes_search = BayesSearchCV(
    estimator=modelo_lgbm_bayes,    # Modelo a ser otimizado
    search_spaces=param_grid,       # Espaço de busca definido acima
    scoring='r2',                   # Critério de seleção
    n_iter=15,                      # Número de avaliações do modelo
    cv=3,                           # Validação cruzada
    random_state=42,                # Semente para reprodutibilidade
    n_jobs=-1,                      # Paralelização total dos cálculos
    verbose=1                       # 0 = sem mensagens, 1 = mensagens de progresso, 2 = mensagens detalhadas
)

In [12]:
fit_params = {
    'eval_metric': ['r2', 'rmse', 'mae'],              # Métricas a serem avaliadas
    'categorical_feature': categorical_features,       # Colunas categóricas
}

In [13]:
# Executar a busca Bayesiana
start_time = time.time()
bayes_search.fit(X_train_final, y_train_final, **fit_params)

# Parar o cronômetro
end_time = time.time()
elapsed_time = end_time - start_time

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[LightGBM] [Info] Total Bins 30558
[LightGBM] [Info] Number of data points in the train set: 487214, number of used

In [14]:
# Melhores parâmetros encontrados
try:
    melhores_parametros = bayes_search.best_params_
    print(f"Melhores parâmetros: {melhores_parametros}")
    print("R2: ", bayes_search.best_score_)
    print(f"Tempo total de execução: {elapsed_time:.2f} segundos")
except:
    melhores_parametros = {
        'colsample_bytree': 0.10290424580379459, 
        'learning_rate': 0.006574004842664949, 
        'max_depth': 104, 
        'n_estimators': 5764, 
        'num_leaves': 37, 
        'reg_alpha': 0.006670287398683358, 
        'reg_lambda': 3.666034360775116e-06, 
        'subsample': 0.22207566947075547}
    print(f"Erro ao obter melhores parâmetros, usando valores calculados anteriormente:\n {melhores_parametros}")

Melhores parâmetros: OrderedDict([('colsample_bytree', 0.14117896200448252), ('learning_rate', 0.0010443208715056967), ('max_depth', 107), ('n_estimators', 7003), ('num_leaves', 60), ('reg_alpha', 0.0014178666840872634), ('reg_lambda', 8.194358602618916e-05), ('subsample', 0.7945549955978624)])
R2:  0.22918088004526949
Tempo total de execução: 14233.65 segundos


In [15]:
# Treinar o modelo com os melhores parâmetros encontrados
modelo_lgbm_bayes.set_params(**melhores_parametros)

start_time = time.time()

# Treinamento do modelo com os melhores parâmetros encontrados
modelo_lgbm_bayes.fit(X_train_final, 
                      y_train_final, 
                      eval_set=[(X_eval, y_eval)],
                      eval_metric=['r2', 'rmse', 'mae'],
                      callbacks=[early_stopping(stopping_rounds=200)],
                      categorical_feature=categorical_features)

tempo_treino = time.time() - start_time

[LightGBM] [Info] Total Bins 30558
[LightGBM] [Info] Number of data points in the train set: 487214, number of used features: 73
[LightGBM] [Info] Start training from score 527.881246
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[7003]	valid_0's rmse: 74.2203	valid_0's l1: 59.0518	valid_0's l2: 5508.65


In [16]:
# Previsões
y_pred_bayes = modelo_lgbm_bayes.predict(X_test)

In [17]:
nome_experimento = 'Escolares CH 2023'

registrar_modelo(experimento=nome_experimento,
                    modelo=modelo_lgbm_bayes,
                    parametros={**modelo_lgbm_bayes.get_params(), "amostra": X_train_final.shape[0], "tempo": tempo_treino},
                    X_train=X_train_final,
                    y_train=y_train_final,
                    y_test=y_test,
                    y_pred=y_pred_bayes,
                    variavel_alvo='NUM_NOTA_CH',
                    nome_modelo='modelo_lgbm_bayes_escolares_2023',
                    descricao_modelo='Modelo LGBMRegressor otimizado com BayesSearchCV para dados escolares CH',)

2025/08/12 01:21:16 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
Registered model 'modelo_lgbm_bayes_escolares_2023' already exists. Creating a new version of this model...
2025/08/12 01:21:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: modelo_lgbm_bayes_escolares_2023, version 3


Modelo registrado com sucesso no MLflow: modelo_lgbm_bayes_escolares_2023
🏃 View run luxuriant-sloth-397 at: http://127.0.0.1:9080/#/experiments/623589849857672017/runs/ffb33448a1e84fb8aec314c20a17249b
🧪 View experiment at: http://127.0.0.1:9080/#/experiments/623589849857672017
Rastreamento do MLflow finalizado.


Created version '3' of model 'modelo_lgbm_bayes_escolares_2023'.


In [18]:
# Avaliação grupo treino
avaliar_modelo(y_train_final, modelo_lgbm_bayes.predict(X_train_final), "treino")

# Avaliação grupo teste
avaliar_modelo(y_test, y_pred_bayes, "teste")

MAE (treino): 58.6596
RMSE (treino): 73.8322
R2 (treino): 0.2389
MAE (teste): 58.7986
RMSE (teste): 73.9859
R2 (teste): 0.2322


In [19]:
# Salvar modelo como Pickle
joblib.dump(modelo_lgbm_bayes, 'Modelos\\modelo_lgbm_bayes_escolares.pkl')

['Modelos\\modelo_lgbm_bayes_escolares.pkl']